# 01 — Data Ingestion

**Project:** The Factory Floor — Industrial Maintenance Copilot  
**Equipment family:** Electric motors + Variable-Frequency Drives (VFDs)

This notebook covers milestone points **1–3**:
1. Choose the equipment family.
2. Collect technical PDFs.
3. Load and extract the documents with useful metadata.

The final corpus will later be expanded to roughly 10–15 manuals. For the Saturday milestone, the goal is to prove that real industrial documentation can be ingested end to end.

## Why this family?

Electric motors and VFDs are tightly related in industrial maintenance. Typical symptoms include overheating, vibration, bearing problems, overcurrent, trips, alarms and parameter/configuration issues. This makes the family broad enough for interesting troubleshooting, but narrow enough for a controlled RAG corpus.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import MANUAL_DIR, SOURCES_CSV

MANUAL_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Manual directory:', MANUAL_DIR)


Project root: C:\Users\se25479\Desktop\Factory_Floor_Chatbot
Manual directory: C:\Users\se25479\Desktop\Factory_Floor_Chatbot\data\manuals


## 1. Inspect the selected official sources

`manual_sources.csv` contains the initial Siemens motor/VFD documentation selected for the milestone. If the PDFs are not yet present, run `python download_manuals.py` from the project root.

> **Important:** the downloader requires internet access. If a manufacturer blocks automated downloads, download the PDF manually from the official page and place it in `data/manuals/`.

In [2]:
import csv

with SOURCES_CSV.open(encoding='utf-8') as f:
    sources = list(csv.DictReader(f))

for row in sources:
    print(f"{row['manufacturer']:8} | {row['family']:28} | {row['filename']}")

Siemens  | SINAMICS G120C               | Siemens_SINAMICS_G120C_Operating_Instructions.pdf
Siemens  | SINAMICS G120C               | Siemens_SINAMICS_G120C_List_Manual.pdf
Siemens  | SINAMICS G120                | Siemens_SINAMICS_G120_Function_Manual.pdf
Siemens  | SINAMICS G120 CU240B-2/CU240E-2 | Siemens_CU240B2_CU240E2_Operating_Instructions.pdf
Siemens  | SINAMICS G120 CU240B/E-2     | Siemens_G120_CU240BE2_List_Manual.pdf
Siemens  | SINAMICS G120                | Siemens_G120_Fieldbus_Function_Manual.pdf
Siemens  | SIMOTICS SD                  | Siemens_SIMOTICS_SD_Operating_Instructions.pdf
Siemens  | SIMOTICS GP/SD/DP            | Siemens_SIMOTICS_GP_SD_DP_Engineering_Manual.pdf
Siemens  | SIMOTICS SD 1LE7             | Siemens_SIMOTICS_SD_1LE7_Operating_Instructions.pdf
Siemens  | SIMOTICS GP 1LE1             | Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf
Siemens  | SIMOTICS GP/SD/XP/DP         | Siemens_SIMOTICS_GP_SD_XP_DP_Catalog.pdf


## 2. Confirm which PDFs are available

In [3]:
pdf_files = sorted(MANUAL_DIR.glob('*.pdf'))
print(f'PDFs found: {len(pdf_files)}')
for p in pdf_files:
    print(f'- {p.name} ({p.stat().st_size / 1_000_000:.2f} MB)')

if not pdf_files:
    print('\nNo PDFs found yet. Run: python download_manuals.py')

PDFs found: 11
- Siemens_CU240B2_CU240E2_Operating_Instructions.pdf (2.69 MB)
- Siemens_G120_CU240BE2_List_Manual.pdf (5.14 MB)
- Siemens_G120_Fieldbus_Function_Manual.pdf (7.22 MB)
- Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf (11.71 MB)
- Siemens_SIMOTICS_GP_SD_DP_Engineering_Manual.pdf (2.49 MB)
- Siemens_SIMOTICS_GP_SD_XP_DP_Catalog.pdf (16.27 MB)
- Siemens_SIMOTICS_SD_1LE7_Operating_Instructions.pdf (2.65 MB)
- Siemens_SIMOTICS_SD_Operating_Instructions.pdf (5.80 MB)
- Siemens_SINAMICS_G120_Function_Manual.pdf (5.85 MB)
- Siemens_SINAMICS_G120C_List_Manual.pdf (4.17 MB)
- Siemens_SINAMICS_G120C_Operating_Instructions.pdf (20.57 MB)


## 3. Load the PDFs page by page

We use `PyPDFLoader`. Loading page by page is useful because every retrieved chunk can keep the original PDF filename and page number as metadata. Those fields will later be used for citations.

In [4]:
%pip install pypdf

from factory_floor.ingestion import load_manuals

documents = load_manuals(MANUAL_DIR)
print(f'Loaded pages: {len(documents)}')


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\se25479\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


Loaded pages: 3679


## 4. Inspect extracted text and metadata

This is a sanity check. We want readable text and metadata before chunking. If a PDF is scanned and the text is empty, it should be replaced with a text-based version or handled separately.

In [5]:
if documents:
    sample = documents[0]
    print('METADATA:')
    print(sample.metadata)
    print('\nTEXT PREVIEW:')
    print(sample.page_content[:1500])

METADATA:
{'producer': 'Adobe PDF Library 11.0', 'creator': 'Acrobat PDFMaker 11 für Word', 'creationdate': '2017-01-18T10:17:27+01:00', 'author': 'Siemens AG, DF MC', 'comments': '', 'company': 'Siemens AG', 'keywords': 'A5E39910322B AA; 01/2017', 'moddate': '2017-01-26T08:33:02+01:00', 'sourcemodified': 'D:20170118091131', 'subject': 'Compact Operating Instructions', 'title': 'CU240B-2 and CU240E-2 Control Units', 'company-long': 'Siemens AG', 'company-short': 'Siemens', 'document-class': 'Compact Operating Instructions', 'document-class-mrl': '', 'edition': '01/2017', 'order-nr': 'A5E39910322B AA', 'print-year': '2015 - 2017', 'product-group': 'SINAMICS G120', 'system': 'SINAMICS', 'source': 'C:\\Users\\se25479\\Desktop\\Factory_Floor_Chatbot\\data\\manuals\\Siemens_CU240B2_CU240E2_Operating_Instructions.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Siemens_CU240B2_CU240E2_Operating_Instructions.pdf', 'manufacturer': 'Siemens', 'equipment_type': 'VFD'}

TEXT

## 5. Corpus summary

In [6]:
from collections import Counter

print('Pages by equipment type:')
print(Counter(d.metadata.get('equipment_type') for d in documents))
print('\nPages by source file:')
for source, count in Counter(d.metadata.get('source_file') for d in documents).most_common():
    print(f'{count:4}  {source}')

Pages by equipment type:
Counter({'VFD': 2766, 'electric_motor': 913})

Pages by source file:
 942  Siemens_G120_CU240BE2_List_Manual.pdf
 758  Siemens_SINAMICS_G120C_List_Manual.pdf
 576  Siemens_SIMOTICS_GP_SD_XP_DP_Catalog.pdf
 500  Siemens_SINAMICS_G120C_Operating_Instructions.pdf
 274  Siemens_SINAMICS_G120_Function_Manual.pdf
 256  Siemens_G120_Fieldbus_Function_Manual.pdf
 140  Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf
 122  Siemens_SIMOTICS_SD_Operating_Instructions.pdf
  44  Siemens_SIMOTICS_SD_1LE7_Operating_Instructions.pdf
  36  Siemens_CU240B2_CU240E2_Operating_Instructions.pdf
  31  Siemens_SIMOTICS_GP_SD_DP_Engineering_Manual.pdf


## Milestone checkpoint

At this point we have:
- a fixed equipment family;
- a reproducible list of official manuals;
- PDF loading and extraction;
- source/page/equipment metadata ready for chunking.

The next notebook converts these pages into chunks, embeddings and a persistent vector database.